# Phase 4 · What one dialysis centre could deliver

> **An illustrative model using public data and stated assumptions. Not investment advice.**

**The question for this phase:** if one haemodialysis centre opened inside a public hospital, how
many patients could it treat, how much of its capacity would it use, and what would it take to
cover its costs?

Terms used below:
- **Capacity used**: the share of available dialysis slots actually filled (0 to 1).
- **Payer mix**: who pays for each session: the patient's family (**out-of-pocket**), the federal
  subsidy, NHIA (national health insurance), or an employer/HMO.
- **Break-even**: the point where money in equals money out (before paying back the set-up cost).
- **Payback**: how many years of surplus it takes to earn back the set-up cost.
- **PPP** (public-private partnership): a private partner and a public hospital share the work
  and the money. The two structures here are **general illustrations**, not any real firm's terms.

Every number below is printed by code. None are typed in.

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
import plotly.io as pio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "pipeline"), str(ROOT / "src")]
import centre_model as M, charts

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 60, "display.width", 180, "display.max_colwidth", 110)

P = ROOT / "data" / "processed"
a = M.load_assumptions()
sc = pd.read_csv(P / "centre_model_scenarios.csv")
sens = pd.read_csv(P / "centre_model_sensitivity.csv")
ppp = pd.read_csv(P / "centre_model_ppp.csv")
print(a["label"])

An illustrative model using public data and stated assumptions. Not investment advice.


## 1. The inputs, and where each comes from

Every input lives in `config/assumptions_kidney.yaml` with a base value, a low and high value, and
a source. Anything I couldn't find a public source for says **"assumption — to verify"**.

In [2]:
rows = []
for section in M.SECTIONS:
    for name, spec in a[section].items():
        rows.append({"input": name, "base": M._resolve(spec["value"]), "low": M._resolve(spec["low"]),
                     "high": M._resolve(spec["high"]), "unit": spec["unit"],
                     "sourced": not str(spec["source"]).startswith("assumption"), "source": spec["source"]})
inputs = pd.DataFrame(rows)
print(f"{inputs.sourced.sum()} of {len(inputs)} inputs have a public source; "
      f"{(~inputs.sourced).sum()} are assumptions to verify.")
inputs[["input", "base", "low", "high", "unit", "sourced"]]

7 of 21 inputs have a public source; 14 are assumptions to verify.


,input,base,low,high,unit,sourced
0,machines,10,6,15,haemodialysis machines,True
1,shifts_per_day,2,2,3,shifts per machine per day,True
2,days_open_per_week,6,5,7,days per week,False
3,capacity_used,0.75,0.5,0.9,share of available session slots used (0-1),False
4,sessions_per_patient_per_week,2,2,3,sessions per patient per week,False
5,price_out_of_pocket_ngn,45000,40000,50000,NGN per session,True
6,price_subsidised_ngn,12000,12000,40000,NGN per session received by the centre,True
7,price_nhia_ngn,40000,30000,45000,NGN per session,False
8,price_employer_hmo_ngn,50000,45000,60000,NGN per session,False
9,consumables_per_session_usd,20,15,30,"US$ per session (dialyser, bloodlines, concentrate, needles; imported)",False


In [3]:
mix = pd.DataFrame({p: {k: v[k] for k in ("value", "low", "high")} for p, v in a["payer_mix"].items()
                    if isinstance(v, dict)}).T
print("Payer mix (share of sessions). Each column sums to:", mix.sum().round(6).to_dict())
mix

Payer mix (share of sessions). Each column sums to: {'value': 1.0, 'low': 1.0, 'high': 1.0}


,value,low,high
out_of_pocket,0.80,0.90,0.65
subsidised,0.10,0.08,0.15
nhia,0.05,0.01,0.10
employer_hmo,0.05,0.01,0.10


### Where the equipment price comes from
Two register sources give a machine price in naira. Converting each at the exchange rate of its
own year (World Bank):

In [4]:
w = pd.read_csv(P / "wdi_nga.csv")
fx = w[w.indicator_id == "PA.NUS.FCRF"].set_index("year").value
for label, naira, machines, year in [("UCH Ibadan: 3 machines for N30m", 30e6, 3, 2019),
                                     ("UCH Ibadan (same, at 2020 rate)", 30e6, 3, 2020),
                                     ("Gombe: 6 machines + water plant for N200m", 200e6, 6, 2017)]:
    print(f"{label:45} -> about US${naira / machines / fx[year]:,.0f} per machine at the {year} rate "
          f"(N{fx[year]:,.0f}/US$)")
print("\nThe Gombe figure includes a water plant, so it overstates the machine price.")
print(f"Model uses US${a['costs']['equipment_cost_per_machine_usd']['value']:,} "
      f"(range {a['costs']['equipment_cost_per_machine_usd']['low']:,}-{a['costs']['equipment_cost_per_machine_usd']['high']:,}).")

UCH Ibadan: 3 machines for N30m               -> about US$32,582 per machine at the 2019 rate (N307/US$)
UCH Ibadan (same, at 2020 rate)               -> about US$27,870 per machine at the 2020 rate (N359/US$)
Gombe: 6 machines + water plant for N200m     -> about US$109,007 per machine at the 2017 rate (N306/US$)

The Gombe figure includes a water plant, so it overstates the machine price.
Model uses US$28,000 (range 20,000-33,000).


## 2. Does the code match a hand calculation?

`tests/test_centre_model.py` holds one case worked out on paper with round numbers: 2 machines,
2 shifts, 5 days, half the slots used, N50,000 a session, N1,000 to the dollar. Running it here:

In [5]:
sys.path.insert(0, str(ROOT / "tests"))
from test_centre_model import hand_case
r = M.run(hand_case())
by_hand = {"weekly_capacity": 20, "patients_treated": 5, "sessions_per_year": 520,
           "variable_cost_per_session_ngn": 15_000, "fixed_cost_per_year_ngn": 12_400_000,
           "net_cash_per_year_ngn": 5_800_000, "breakeven_capacity_used": 0.3407,
           "breakeven_price_ngn": 38_846.15, "payback_years": 4.31, "cost_per_patient_year_ngn": 5_040_000}
check = pd.DataFrame({"by hand": by_hand, "code": {k: round(r[k], 4) for k in by_hand}})
check["match"] = [math.isclose(h, c, rel_tol=1e-3) for h, c in zip(check["by hand"], check["code"])]
print("All match:", check.match.all())
check

All match: True


,by hand,code,match
weekly_capacity,2.000000e+01,2.000000e+01,True
patients_treated,5.000000e+00,5.000000e+00,True
sessions_per_year,5.200000e+02,5.200000e+02,True
variable_cost_per_session_ngn,1.500000e+04,1.500000e+04,True
fixed_cost_per_year_ngn,1.240000e+07,1.240000e+07,True
net_cash_per_year_ngn,5.800000e+06,5.800000e+06,True
breakeven_capacity_used,3.407000e-01,3.407000e-01,True
breakeven_price_ngn,3.884615e+04,3.884615e+04,True
payback_years,4.310000e+00,4.310300e+00,True
cost_per_patient_year_ngn,5.040000e+06,5.040000e+06,True


## 3. Patients treated and break-even, low / base / high

- **Low** puts every uncertain input at its pessimistic end (fewer slots used, lower prices,
  higher costs). **High** does the opposite.
- Design choices (number of machines, shifts, days open, sessions per patient) stay the same in
  all three, so the scenarios show uncertainty, not a different centre.
- Each scenario is also run with the naira **20%** and **50%** weaker, because consumables and
  machines are imported and priced in dollars, while patients pay in naira.

In [6]:
charts.centre_scenarios(sc).show()

In [7]:
show = sc[["scenario", "naira_shock", "patients_treated", "sessions_per_year", "capacity_used",
           "average_price_ngn", "breakeven_price_ngn", "breakeven_capacity_used",
           "net_cash_per_year_ngn", "payback_years", "cost_per_patient_year_ngn"]].copy()
show["breakeven_capacity_used"] = show.breakeven_capacity_used.map(
    lambda v: "not reachable" if not math.isfinite(v) or v > 1 else f"{v:.0%}")
show["payback_years"] = show.payback_years.map(lambda v: "never (loss-making)" if not math.isfinite(v) else f"{v:.1f}")
for c in ["average_price_ngn", "breakeven_price_ngn", "net_cash_per_year_ngn", "cost_per_patient_year_ngn"]:
    show[c] = show[c].map(lambda v: f"{'−' if v < 0 else ''}N{abs(v):,.0f}")
show

,scenario,naira_shock,patients_treated,sessions_per_year,capacity_used,average_price_ngn,breakeven_price_ngn,breakeven_capacity_used,net_cash_per_year_ngn,payback_years,cost_per_patient_year_ngn
0,low,0.0,30.0,3120.0,0.50,"N36,950","N115,641",not reachable,"−N245,516,314",never (loss-making),"N17,391,623"
1,low,0.2,30.0,3120.0,0.50,"N36,950","N129,569",not reachable,"−N288,972,376",never (loss-making),"N19,913,148"
2,low,0.5,30.0,3120.0,0.50,"N36,950","N150,462",not reachable,"−N354,156,471",never (loss-making),"N23,695,435"
3,base,0.0,45.0,4680.0,0.75,"N41,700","N61,452",not reachable,"−N92,439,123",never (loss-making),"N8,319,106"
4,base,0.2,45.0,4680.0,0.75,"N41,700","N69,342",not reachable,"−N129,366,148",never (loss-making),"N9,525,327"
5,base,0.5,45.0,4680.0,0.75,"N41,700","N81,178",not reachable,"−N184,756,685",never (loss-making),"N11,334,658"
6,high,0.0,54.0,5616.0,0.90,"N49,250","N39,450",51%,"N55,038,256",7.7,"N4,890,082"
7,high,0.2,54.0,5616.0,0.90,"N49,250","N44,762",67%,"N25,205,107",20.2,"N5,600,009"
8,high,0.5,54.0,5616.0,0.90,"N49,250","N52,730",not reachable,"−N19,544,617",never (loss-making),"N6,664,900"


In [8]:
b = sc[(sc.scenario == "base") & (sc.naira_shock == 0)].iloc[0]
print(f"Base case: {b.patients_treated:.0f} people on dialysis, {b.sessions_per_year:,.0f} sessions a year, "
      f"{b.capacity_used:.0%} of slots used.")
print(f"It receives N{b.average_price_ngn:,.0f} a session on average, and needs N{b.breakeven_price_ngn:,.0f} to break even.")
print(f"Of each session's cost, N{b.variable_cost_per_session_ngn:,.0f} is consumables plus power and water, "
      f"which leaves N{b.average_price_ngn - b.variable_cost_per_session_ngn:,.0f} towards staff and maintenance.")
print(f"Exchange rate used: N{b.fx_ngn_per_usd:,.0f} per US$ (World Bank, {int(b.fx_year)} average).")

Base case: 45 people on dialysis, 4,680 sessions a year, 75% of slots used.
It receives N41,700 a session on average, and needs N61,452 to break even.
Of each session's cost, N36,368 is consumables plus power and water, which leaves N5,332 towards staff and maintenance.
Exchange rate used: N1,518 per US$ (World Bank, 2025 average).


## 4. Which assumptions matter most?

The model is only as good as its inputs, and many are assumptions. This moves one input at a time
to its low and high value and shows how far yearly net cash moves. The longest bars are the
inputs an on-the-ground assessment should pin down first.

In [9]:
charts.centre_sensitivity(sens).show()

In [10]:
top = sens.head(5)
print("Verify these first (biggest effect on the result):")
for _, r in top.iterrows():
    print(f"  {r.input:40} swing N{r.swing_ngn / 1e6:,.0f}m a year  "
          f"({'has a public source' if r.verified else 'ASSUMPTION, no public source'})")

Verify these first (biggest effect on the result):
  consumables_per_session_usd              swing N107m a year  (ASSUMPTION, no public source)
  staff_cost_per_shift_ngn                 swing N62m a year  (ASSUMPTION, no public source)
  price_out_of_pocket_ngn                  swing N37m a year  (has a public source)
  maintenance_pct_of_equipment_per_year    swing N34m a year  (ASSUMPTION, no public source)
  power_water_per_session_ngn              swing N28m a year  (ASSUMPTION, no public source)


## 5. What this means for Halima

Halima needs dialysis two or three times a week, for life. What her family would pay in a year,
at the prices in the register:

In [11]:
p = a["prices"]
for sessions in (2, 3):
    full = p["price_out_of_pocket_ngn"]["value"] * sessions * 52
    sub = p["price_subsidised_ngn"]["value"] * sessions * 52
    print(f"{sessions} sessions a week: N{full:,.0f} a year at N{p['price_out_of_pocket_ngn']['value']:,} a session; "
          f"N{sub:,.0f} at the subsidised N{p['price_subsidised_ngn']['value']:,}")
print(f"\nCost to the centre of treating one patient for a year (base case): N{b.cost_per_patient_year_ngn:,.0f}")
print("Her family's income is not in this dataset, so I can't yet say what share of it this is.")

2 sessions a week: N4,680,000 a year at N45,000 a session; N1,248,000 at the subsidised N12,000
3 sessions a week: N7,020,000 a year at N45,000 a session; N1,872,000 at the subsidised N12,000

Cost to the centre of treating one patient for a year (base case): N8,319,106
Her family's income is not in this dataset, so I can't yet say what share of it this is.


## 6. Two illustrative PPP structures

> **Illustrative structures, general examples only. Not any real firm's terms.**

In both, the private partner pays for equipment, set-up, maintenance and consumables; the host
hospital pays for staff, power and water.
- **(a) Fee per session:** the hospital pays the partner a fixed fee for every session.
- **(b) Revenue share:** the partner receives a share of what each session brings in.

In [12]:
t = ppp.copy()
t["naira_shock"] = t.naira_shock.map("{:.0%}".format)
for c in [c for c in t.columns if c.endswith("_ngn")]:
    t[c] = t[c].map(lambda v: f"{'−' if v < 0 else ''}N{abs(v) / 1e6:,.1f}m")
t["partner_payback_years"] = t.partner_payback_years.map(lambda v: "never" if not math.isfinite(v) else f"{v:.0f}")
print("Terms used: fee N{:,} a session; revenue share {:.0%}".format(
    a["ppp"]["partner_fee_per_session_ngn"]["value"], a["ppp"]["partner_revenue_share"]["value"]))
t.drop(columns="label")

Terms used: fee N40,000 a session; revenue share 95%


,naira_shock,structure,partner_income_ngn,partner_costs_ngn,partner_net_cash_ngn,partner_payback_years,hospital_income_ngn,hospital_costs_ngn,hospital_net_cash_ngn
0,0%,fee,N187.2m,N184.6m,N2.6m,237,N8.0m,N103.0m,−N95.0m
1,20%,fee,N187.2m,N221.6m,−N34.4m,never,N8.0m,N103.0m,−N95.0m
2,50%,fee,N187.2m,N277.0m,−N89.8m,never,N8.0m,N103.0m,−N95.0m
3,0%,share,N185.4m,N184.6m,N0.8m,796,N9.8m,N103.0m,−N93.2m
4,20%,share,N185.4m,N221.6m,−N36.2m,never,N9.8m,N103.0m,−N93.2m
5,50%,share,N185.4m,N277.0m,−N91.6m,never,N9.8m,N103.0m,−N93.2m


At today's base-case prices, the money from each session isn't enough to cover both partners'
costs, whichever structure is used. A structure moves the shortfall between the partner and the
hospital; it doesn't remove it. Covering it needs a higher price per session, a larger insured or
subsidised share at a fuller tariff, lower imported costs, or outside funding.

## 7. Try your own inputs
Change any value and re-run. The payer mix must still add up to 1.

In [13]:
x = M.scenario_inputs(a, "base")
x.update({
    "machines": 10,
    "capacity_used": 0.75,
    "price_out_of_pocket_ngn": 45_000,
    "consumables_per_session_usd": 20,
})
# x["payer_mix"] = M.payer_mix_column(a, "high")   # try the 'more insured' mix
mine = M.run(x, naira_shock=0.0)
print(f"Patients treated: {mine['patients_treated']:.0f}")
print(f"Price received: N{mine['average_price_ngn']:,.0f} · break-even price: N{mine['breakeven_price_ngn']:,.0f}")
v = mine["net_cash_per_year_ngn"]
print(f"Net cash a year: {'−' if v < 0 else ''}N{abs(v):,.0f}")
print("Payback:", "never" if not math.isfinite(mine["payback_years"]) else f"{mine['payback_years']:.1f} years")

Patients treated: 45
Price received: N41,700 · break-even price: N61,452
Net cash a year: −N92,439,123
Payback: never


## 8. Checks

In [14]:
import validate
errors = validate.check_assumptions() + validate.check_payer_mix()
print("Assumption checks passed: every input has value, low, high, unit, source; payer mixes sum to 1."
      if not errors else errors)

Assumption checks passed: every input has value, low, high, unit, source; payer mixes sum to 1.
